# Lab 04 — Agentic RAG and fine-tuning decisions (solution)

> Reference solution: every TODO is filled in and every check passes.

**Level:** Advanced · **Duration:** ~55 minutes · **Covers:** H7 agentic RAG: the retriever as a tool (25 min) · fine-tuning decision workshop, 4 scenarios (30 min, from the delivery schedule)

Lab guide: [`../lab-04-agentic-rag-fine-tuning-decisions.md`](../lab-04-agentic-rag-fine-tuning-decisions.md) — read each part there first; this notebook holds the code.

You turn the Lab 03 retrieval into a tool that a LlamaIndex agent calls only when it needs to, measure it against the fixed 2-step pipeline on both providers, then decide fine-tuning vs RAG vs prompting vs hybrid for four Daimler Truck scenarios and defend each decision. You design, measure and justify.

In [ ]:
# (notebook) Setup: run this cell first. LLM_PROVIDER picks the model, EMBEDDING_PROVIDER the embeddings.
import json
import os
import sys
import time
import warnings
from pathlib import Path

import pandas as pd
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(usecwd=True))  # walks up from this folder to the repo-root .env
os.environ.setdefault("ANONYMIZED_TELEMETRY", "False")  # Chroma: no usage telemetry
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message="IProgress not found")
pd.set_option("display.max_colwidth", 110)

# labs/module-03 holds data/ and rag_kit.py; the notebook runs from notebooks/ or solutions/.
LAB_DIR = next(p for p in (Path.cwd(), *Path.cwd().parents) if p.name == "module-03" and (p / "data").is_dir())
DATA_DIR, OUTPUT_DIR = LAB_DIR / "data", LAB_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
sys.path[:0] = [str(LAB_DIR.parent / "setup"), str(LAB_DIR)]
import rag_kit as rk  # noqa: E402
from llm import (  # noqa: E402
    describe, embedding_slug, get_embedding_model_name, get_embedding_provider, get_provider,
    is_configured, partner_provider,
)
from transformers.utils import logging as hf_logging  # noqa: E402

hf_logging.disable_progress_bar()
PROVIDER, PARTNER = get_provider(), partner_provider()
if not is_configured(PROVIDER):
    raise SystemExit(
        f"❌ LLM_PROVIDER={PROVIDER} is not configured. Add its key and model to the repo-root .env "
        "(template: labs/setup/.env.example), restart the kernel and run this cell again."
    )
HAS_PARTNER = is_configured(PARTNER)


def check(name: str, fn) -> None:
    """Run a check function. A returned string means 'skipped'; a TODO is reported, not raised."""
    try:
        skipped = fn()
    except NotImplementedError as exc:
        print(f"⏳ {name}: TODO not implemented yet ({exc})")
        return
    print(f"⚠️  {name}: skipped ({skipped})" if skipped else f"✅ {name}")


print(describe())
print(f"Comparison provider: {PARTNER} {'✅' if HAS_PARTNER else '⚠️  not configured: comparison cells skip it'}")
print(f"Embeddings: {get_embedding_provider()} · {get_embedding_model_name()} (index suffix {embedding_slug()})")

from dataclasses import asdict, dataclass

from llama_index.core import PromptTemplate, QueryBundle
from llama_index.core.agent.workflow import FunctionAgent, ToolCallResult
from pydantic import BaseModel, Field

rk.setup_llama(PROVIDER)
index, nodes = rk.build_knowledge_base(f"lab04-{embedding_slug()}")  # corpus + SharePoint + Confluence, on Chroma
USER = rk.EU_AFTERSALES  # the authenticated caller: an EU after-sales engineer
print(f"{len(nodes)} nodes · user {USER.name} sees {sum(rk.visible_to(USER, n.metadata) for n in nodes)} of them")

## Part 1 — The retriever as a tool (10 min)

A tool is an RPC contract: the docstring and type hints become the schema the model reads. Keep
the input narrow, return compact JSON with stable IDs, and say what to do when nothing is found.

In [ ]:
# (notebook) Given: hybrid search + re-rank + score floor, inside the user's access filters (Labs 02-03).
DOC_TYPES = ("bulletin", "kb", "faq", "runbook", "policy", "contract", "engineering")
ALLOWED = [n for n in nodes if rk.visible_to(USER, n.metadata)]
RERANK = rk.reranker(4)


def hybrid_search(query: str, doc_type: str = "any", floor: float = -1.0):
    """Up to 4 re-ranked nodes above the floor, for one doc_type or "any"."""
    types = None if doc_type == "any" else [doc_type]
    pool = [n for n in ALLOWED if types is None or n.metadata["doc_type"] in types]
    candidates = rk.hybrid_retriever(index, pool, k=10, pool=10, filters=rk.access_filters(USER, types)).retrieve(query)
    return [h for h in RERANK.postprocess_nodes(candidates, query_bundle=QueryBundle(query)) if h.score >= floor]

In [ ]:
# (notebook) Your TODO: the tool contract. The docstring IS the description the model sees.
def search_knowledge(query: str, doc_type: str = "any") -> str:
    """Search Daimler Truck workshop bulletins (bulletin), repair guides (kb), customer FAQs (faq),
    IT runbooks (runbook), policies (policy), contracts (contract) and engineering specs (engineering).
    Use it for any question about vehicles, fault codes, repairs, warranty, policies or IT procedures;
    do not use it for greetings or thanks. doc_type narrows the search to one of those types, or "any".
    Returns up to 4 passages as JSON with doc_id, title and text."""
    if doc_type != "any" and doc_type not in DOC_TYPES:
        return f"NO_RESULTS: unknown doc_type {doc_type!r}. Use one of {', '.join(DOC_TYPES)} or 'any'."
    hits = hybrid_search(query, doc_type)
    if not hits:
        return "NO_RESULTS: nothing relevant found. Rephrase the query, or try doc_type='any'."
    return json.dumps([{"doc_id": h.node.metadata["doc_id"], "title": h.node.metadata["title"],
                        "text": h.node.get_content()} for h in hits], ensure_ascii=False)

In [ ]:
# (notebook) Check: JSON with stable ids, the doc_type filter works, and misses say what to do next.
def check_part1():
    passages = json.loads(search_knowledge("FC-BAT-2201 threshold", doc_type="engineering"))
    assert passages and {p["doc_id"] for p in passages} == {"ENG-SPEC-0417"}, passages
    assert {"doc_id", "title", "text"} <= set(passages[0])
    assert search_knowledge("FC-BAT-2201", doc_type="manual").startswith("NO_RESULTS")
    assert search_knowledge("What is the canteen menu on Friday?").startswith("NO_RESULTS")


check("Part 1 — search_knowledge tool", check_part1)
tool_fn = rk.todo_or_reference(search_knowledge, rk.make_search_tool(index, nodes, USER), "FC-BAT-2201")

## Part 2 — An agent that decides when and where to search (15 min)

**2-step RAG** retrieves once for every question. **Agentic RAG** lets the model decide whether to
search, for what, in which `doc_type`, and whether to search again.

In [ ]:
# (notebook) A LlamaIndex FunctionAgent with one tool, and a trace of every search it makes.
SYSTEM = (
    "You are the Daimler Truck after-sales assistant for workshop engineers. Search before you answer any "
    "question about vehicles, fault codes, repairs, warranty, policies or IT procedures, and cite doc_ids in "
    "square brackets. Answer only from what the search returned. Search at most 3 times per question."
)
TASKS = {
    "thanks": ("Thanks, that fixed it!", set()),
    "two-part": ("An ED-26 shows FC-BAT-2201 during DC fast charging. Above which temperature spread is the "
                 "charging power limited, and is a battery module replacement covered by warranty on a "
                 "5-year-old truck?", {"POL-WAR-007"}),
    "German": ("Ein OTA-Update hängt in der Werkstatt bei 60 %. Was soll ich tun?", {"TSB-2026-052"}),
}


async def run_agent(provider: str, task: str) -> dict:
    agent = FunctionAgent(tools=[rk.agent_tool(tool_fn, "search_knowledge")],  # runs on the event loop
                          llm=rk.llama_llm(provider), system_prompt=SYSTEM)
    searches, returned = [], set()
    with rk.Timer() as t:
        handler = agent.run(user_msg=task)
        async for event in handler.stream_events():
            if isinstance(event, ToolCallResult):
                searches.append(event.tool_kwargs)
                returned |= rk.cited_ids(str(event.tool_output.content))
        answer = str(await handler)
    return {"searches": searches, "returned": returned, "answer": answer, "s": t.seconds}


agent_runs = []
for provider in [PROVIDER, PARTNER] if HAS_PARTNER else [PROVIDER]:
    for name, (task, expected) in TASKS.items():
        run = await run_agent(provider, task)
        cited = rk.cited_ids(run["answer"])
        agent_runs.append({"provider": provider, "task": name, "searches": len(run["searches"]),
                           "queries": [s.get("query", "")[:40] + f" ({s.get('doc_type', 'any')})" for s in run["searches"]],
                           "cited": sorted(cited), "cited ⊆ returned": cited <= run["returned"],
                           "expected cited": expected <= cited, "s": run["s"], "answer": run["answer"]})
agent_runs = pd.DataFrame(agent_runs)
agent_runs.drop(columns="answer")

In [ ]:
# (notebook) The two-part answer, and the fixed 2-step pipeline on the same tasks for comparison.
print(agent_runs.query("task == 'two-part'").iloc[0]["answer"], "\n")
two_step = rk.make_assistant(rk.RagProfile("2-step", DOC_TYPES, "You are the Daimler Truck after-sales assistant."), USER, index)
rows = []
for name, (task, expected) in TASKS.items():
    with rk.Timer() as t:
        response = two_step.query(task)
    cited = rk.cited_ids(str(response))
    rows.append({"task": name, "searches": 1, "answered": rk.answered(response), "cited": sorted(cited),
                 "expected cited": expected <= cited, "s": t.seconds})
pd.DataFrame(rows)

In [ ]:
# (notebook) Check: no search for small talk; every cited id came from a search; the two-part task found the warranty.
def check_part2():
    for _, run in agent_runs.iterrows():
        assert run["cited ⊆ returned"], (run["provider"], run["task"], run["cited"])
    assert (agent_runs.query("task == 'thanks'")["searches"] == 0).all(), "the agent searched for 'thanks'"
    assert agent_runs.query("task == 'two-part'")["expected cited"].all(), "the two-part answer missed POL-WAR-007"


check("Part 2 — agentic RAG", check_part2)

📝 **Decision (2–3 sentences).** For the workshop assistant, would you ship the agent or the 2-step
pipeline? Use your table: searches, cited documents, seconds, and the "thanks" row.

> *Your answer here.*

## Part 3 — Fine-tuning decision workshop: four scenarios (20 min, from the delivery schedule)

In teams, read the four scenario cards in [`templates/scenario-cards.md`](../templates/scenario-cards.md)
and fill in one canvas per scenario ([`templates/fine-tuning-decision-canvas.md`](../templates/fine-tuning-decision-canvas.md)).
First question every time: is the gap **knowledge** or **behaviour**?

In [ ]:
# (notebook) Your TODO: one canvas per scenario. Decisions: "prompt engineering", "RAG", "fine-tuning", "hybrid".
DECISIONS = {"prompt engineering", "RAG", "fine-tuning", "hybrid"}


@dataclass
class Canvas:
    scenario: str
    gap: str              # "knowledge" or "behaviour"
    decision: str         # one of DECISIONS
    why: str              # data availability, latency, cost, update frequency
    overturn_eval: str    # the evaluation result that would make you change the decision


CANVASES = [
    Canvas("A · Workshop repair assistant", "knowledge", "RAG",
           "3,000 bulletins and guides change weekly and technicians must see the source; re-indexing takes minutes, "
           "retraining weeks, and weights cannot cite or forget a withdrawn bulletin.",
           "Recall@4 below 0.9 on 300 labelled workshop questions after hybrid retrieval and re-ranking."),
    Canvas("B · Warranty claim coding", "behaviour", "fine-tuning",
           "A fixed set of 380 damage codes, 200,000 labelled historical claims, 25,000 claims a month and a 300 ms "
           "budget on dealer systems: a small fine-tuned open-weight classifier beats a large model on cost and "
           "latency, after a few-shot baseline on the small tier sets the bar.",
           "The few-shot small-tier baseline reaches the target accuracy and p95 latency on a held-out month of claims."),
    Canvas("C · Goodwill decision letters", "behaviour", "prompt engineering",
           "Tone, structure and the 'goodwill is not warranty' rule fit in a template with three examples per language; "
           "the facts come from the case; 40 letters a day make cost irrelevant.",
           "Reviewers reject more than 10% of letters for tone in two languages after two prompt iterations."),
    Canvas("D · Engineering specification assistant", "knowledge", "hybrid",
           "Specs change weekly and need exact values with citations (RAG), but part codes and German-English jargon "
           "defeat dense retrieval: hybrid BM25 + vector retrieval first, and a RAFT-style fine-tune only if retrieval "
           "is right but answers still misread the domain.",
           "With the right passages retrieved, answer accuracy on 400 engineer Q&A pairs is already above 95%: then skip the fine-tune."),
]

In [ ]:
# (notebook) Check: four complete canvases, each with a valid decision and an evaluation that could overturn it.
def check_part3():
    if any("TODO" in asdict(c).values() for c in CANVASES):
        raise NotImplementedError("fill in all four canvases")
    for c in CANVASES:
        assert c.gap in {"knowledge", "behaviour"} and c.decision in DECISIONS, c.scenario
        assert len(c.overturn_eval) > 20, f"{c.scenario}: name a measurable evaluation"


check("Part 3 — decision canvases", check_part3)
pd.DataFrame([asdict(c) for c in CANVASES])[["scenario", "gap", "decision"]]

## Part 4 — Challenge your decisions with a second opinion (10 min)

A reviewer model, on both providers, argues against each canvas. You decide whether it changes anything.

In [ ]:
# (notebook) A structured critique per canvas, from the small tier of each provider.
class Critique(BaseModel):
    agrees: bool = Field(description="True if the decision is the best of the four options")
    strongest_counterargument: str = Field(description="One sentence")
    eval_to_run_first: str = Field(description="The single cheapest evaluation that would settle it")


CRITIC = PromptTemplate(
    "You review fine-tuning decisions. Options: prompt engineering, RAG, fine-tuning, hybrid. Note: the "
    "OpenAI fine-tuning API is being wound down and Claude cannot be fine-tuned, so fine-tuning means an "
    "open-weight model the company hosts.\n\nSCENARIO: {scenario}\nGAP: {gap}\nDECISION: {decision}\n"
    "WHY: {why}\nWOULD BE OVERTURNED BY: {overturn_eval}"
)
if any("TODO" in asdict(c).values() for c in CANVASES):
    print("⏳ Fill in the four canvases in Part 3 first.")
else:
    critiques = []
    for provider in [PROVIDER, PARTNER] if HAS_PARTNER else [PROVIDER]:
        critic = rk.llama_llm(provider, small=True)
        for c in CANVASES:
            verdict = critic.structured_predict(Critique, CRITIC, **asdict(c))
            critiques.append({"critic": provider, "scenario": c.scenario[:30], "decision": c.decision, **verdict.model_dump()})
    critiques = pd.DataFrame(critiques)
    display(critiques[["critic", "scenario", "decision", "agrees", "strongest_counterargument"]])

📝 **Write-up (3–5 sentences).** Which critique was strongest, and did it change a decision? For the
scenario your team presents, state the decision and the evaluation that would overturn it.

> *Your answer here.*

---

**Done.** Answer the checkpoint questions in the lab guide. **Next module:** *Module 4 — Multi-Agent
Orchestration with LangGraph, Agentic Patterns & MCP*, where this retriever tool becomes one
specialist in a supervised team of agents.